In [ ]:
# import os
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.animation as animation
# import sys

# sys.path.append(os.path.abspath(".."))
# import Wave2D as W2D

# sol =W2D.Wave2D_Neumann()
# cfl= 1.0 / np.sqrt(2)
# N= 40
# Nt = 60

# data= sol(N=N, Nt=Nt, cfl=cfl, mx=2, my=2, store_data=2)
# x= np.linspace(0, sol.L, N + 1)
# y= np.linspace(0, sol.L, N + 1)
# xij, yij = np.meshgrid(x, y, indexing="ij")

# fig, ax= plt.subplots(subplot_kw={"projection": "3d"}, figsize=(6, 5), dpi=80)
# ax.set_zlim(-1.5, 1.5)
# ax.set_xlabel("x")
# ax.set_ylabel("y")
# ax.set_zlabel("u")

# frames= []
# for n, val in data.items():
#     frame= ax.plot_wireframe(xij, yij, val, rstride=2, cstride=2, color="royalblue")
#     frames.append([frame])

# ani= animation.ArtistAnimation(fig, frames, interval=100, blit=True, repeat_delay=1000)

# #os.makedirs("report", exist_ok=True)
# #output_path = os.path.join("report", "neumannwave.gif")
# output_path= "neumannwave.gif"
# ani.save(output_path, writer="pillow", fps=10)
# plt.close(fig)

# file_size_kb = os.path.getsize(output_path) / 1024
# print(f"Saved:{output_path}")
# print(f"Size:{file_size_kb:.2f} KB")
# assert file_size_kb < 1024, "File size exceeds 1 MB, bring down N and/or Nt!"


--- [MESH GENERATION] ---
Generated 2D grid for N=40 (41x41 nodes).

--- [MESH GENERATION] ---
Generated 2D grid for N=40 (41x41 nodes).
Saved: report/neumannwave.gif
Size:  598.01 KB


### 1.2.3 Exact solution

To demonstrate that Eq (1.6) satisfies the wave eqn, we need to re-express its partial derivatives and sub them back into the standard 2D continuous wave eqn: $\frac{\partial^2 u}{\partial t^2} = c^2 \left( \frac{\partial^2 u}{\partial x^2} + \frac{\partial^2 u}{\partial y^2} \right)$

Let's start wit the given complex exponential function $u(t, x, y) = e^{\imath(k_x x + k_y y - \omega t)}$. Applying the chain rule, we differentiate the outer exponential function, then multiply by the derivative of the inner exponent with respect to $t$. This pulls down the constant multiplier $-\imath\omega$:   

$$\frac{\partial u}{\partial t} = e^{\imath(k_x x + k_y y - \omega t)} \cdot \frac{\partial}{\partial t}(\imath k_x x + \imath k_y y - \imath \omega t) = u \cdot (-\imath\omega) = -\imath\omega u$$

Diffing a second time yields:

$$\frac{\partial^2 u}{\partial t^2} = -\imath\omega \left( \frac{\partial u}{\partial t} \right) = (-\imath\omega)(-\imath\omega) u = \imath^2 \omega^2 u$$

The second time deriv is further simplified to:   

$$\frac{\partial^2 u}{\partial t^2} = -\omega^2 u$$

Next, we apply the exact same chain rule process for the spatial dims $x$ and $y$. For $x$, the first derivative pulls down the constant $\imath k_x$:

$$\frac{\partial u}{\partial x} = e^{\imath(k_x x + k_y y - \omega t)} \cdot \frac{\partial}{\partial x}(\imath k_x x + \imath k_y y - \imath \omega t) = u \cdot (\imath k_x) = \imath k_x u$$

$$\frac{\partial^2 u}{\partial x^2} = \imath k_x \left( \frac{\partial u}{\partial x} \right) = (\imath k_x)^2 u = \imath^2 k_x^2 u = -k_x^2 u$$

For $y$, the first and second derivatives pull down the constant $\imath k_y$:

$$\frac{\partial u}{\partial y} = e^{\imath(k_x x + k_y y - \omega t)} \cdot \frac{\partial}{\partial y}(\imath k_x x + \imath k_y y - \imath \omega t) = u \cdot (\imath k_y) = \imath k_y u$$

$$\frac{\partial^2 u}{\partial y^2} = \imath k_y \left( \frac{\partial u}{\partial y} \right) = (\imath k_y)^2 u = \imath^2 k_y^2 u = -k_y^2 u$$

Having obtained the derivatives, we plug them back into the general wave eqn:

$$-\omega^2 u = c^2 \left(-k_x^2 u - k_y^2 u \right)$$

After factoring out the $-u$ on the RHS and dividing both sides by $-u$:

$$\omega^2 = c^2(k_x^2 + k_y^2)$$

Lastyl, we take the square root to obtain the the exact continuous dispersion relation:

$$\omega = c\sqrt{k_x^2 + k_y^2}$$

Since substituting the Eq(1.6) resolves into the exact continuous dispersion relation, it proves that the complex exponential function is a valid stationary solution to the wave equation, with its real and imaginary components representing the physical waves

### 1.2.4 Dispersion coefficient

Show that the discrete wave eqn gives the true angular frequency ($\omega$) under the assumption of the CFL condition $C = 1/\sqrt{2}$, meaning $\tilde{\omega} = \omega = ck\sqrt{2}$.

For a wave traveling diagonally where $k_x = k_y = k$, the continuous dispersion relation simplifies to:

$$\omega = c \|\mathbf{k}\|_2 = c \sqrt{k_x^2 + k_y^2} = c \sqrt{k^2 + k^2} = ck\sqrt{2}$$

When we simulate this on a grid, the computer spits out a numerical frequency ($\tilde{\omega}$). We want to prove that if we set $C = 1/\sqrt{2}$, the numerical and exact frequencies perfectly match...

First, take the discrete mesh function $u_{i,j}^n = e^{\imath(kh(i+j) - \tilde{\omega}n\Delta t)}$ and plug it into LHS of Eq. 1.3; $u^{n+1}_{i,j} - 2u^n_{i,j} + u^{n-1}_{i,j}$:

Let's define $E = \tilde{\omega}\Delta t$ to save some writing. Expanding out the terms:

$$u_{i,j}^{n+1} = u_{i,j}^n e^{-\imath E}$$
$$-2u_{i,j}^n = -2u_{i,j}^n$$
$$u_{i,j}^{n-1} = u_{i,j}^n e^{\imath E}$$

Combining them yields:

$$u_{i,j}^n e^{-\imath E} - 2u_{i,j}^n + u_{i,j}^n e^{\imath E}$$

Factoring out $u_{i,j}^n$:

$$u_{i,j}^n \left( e^{-\imath E} - 2 + e^{\imath E} \right)$$

Using Euler's formula ($e^{\imath E} + e^{-\imath E} = 2\cos(E)$):

$$u_{i,j}^n \left( 2\cos(E) - 2 \right) = -2u_{i,j}^n \left( 1 - \cos(E) \right)$$

Applying the half-angle trig identy $1 - \cos(E) = 2\sin^2\left(\frac{E}{2}\right)$:

$$-4u_{i,j}^n \sin^2\left(\frac{E}{2}\right) = -4u_{i,j}^n \sin^2\left(\frac{\tilde{\omega}\Delta t}{2}\right)$$

Now we do the exact same thing to the RHS spatial diffs in Eq 1.3.

For the $x$-direction ($i+1$ and $i-1$):

$$u^n_{i+1,j} - 2u^n_{i,j} + u^n_{i-1,j} = u_{i,j}^n \left( e^{\imath kh} - 2 + e^{-\imath kh} \right) = -4u_{i,j}^n \sin^2\left(\frac{kh}{2}\right)$$

Since the wavenumbers are equal in both directions ($k_x = k_y = k$), the x and y spatial derivatives are basically identical. So for the $y$-direction ($j+1$ and $j-1$):

$$u^n_{i,j+1} - 2u^n_{i,j} + u^n_{i,j-1} = u_{i,j}^n \left( e^{\imath kh} - 2 + e^{-\imath kh} \right) = -4u_{i,j}^n \sin^2\left(\frac{kh}{2}\right)$$

Plugging these LHS and RHS expressions back into the main Eq 1.3 gives us:

$$\frac{-4u_{i,j}^n \sin^2\left(\frac{\tilde{\omega}\Delta t}{2}\right)}{\Delta t^2} = c^2 \left( \frac{-4u_{i,j}^n \sin^2\left(\frac{kh}{2}\right)}{h^2} + \frac{-4u_{i,j}^n \sin^2\left(\frac{kh}{2}\right)}{h^2} \right)$$

Divide both sides by $-4u_{i,j}^n$ and then multiply out by $\Delta t^2$ to obtain:

$$\sin^2\left(\frac{\tilde{\omega}\Delta t}{2}\right) = 2 \frac{c^2 \Delta t^2}{h^2} \sin^2\left(\frac{kh}{2}\right)$$

Remember that the 2D Courant number is defined as $C = c\frac{\Delta t}{h}$; sub in $C^2$:

$$\sin^2\left(\frac{\tilde{\omega}\Delta t}{2}\right) = 2 C^2 \sin^2\left(\frac{kh}{2}\right)$$

Forcing $C = 1/\sqrt{2}$ $\implies$ $C^2 = 1/2$. Plugging this in and crossing out redundant terms yields:

$$\sin^2\left(\frac{\tilde{\omega}\Delta t}{2}\right) = \sin^2\left(\frac{kh}{2}\right)$$

Assuming small step sizes, we can take the root of both sides and just equte the sine arguments directly:

$$\frac{\tilde{\omega}\Delta t}{2} = \frac{kh}{2} \implies \tilde{\omega}\Delta t = kh \implies \tilde{\omega} = k\frac{h}{\Delta t}$$

Now, just rearrange the original CFL conditon $C = c\frac{\Delta t}{h} = 1/\sqrt{2}$ to solve for $\frac{h}{\Delta t}$:

$$\frac{h}{\Delta t} = c\sqrt{2}$$

Subbing this ratio back into our $\tilde{\omega}$ equation yields:

$$\tilde{\omega} = k(c\sqrt{2}) = ck\sqrt{2}$$

Since we already know the continuous exact frequency is $\omega = ck\sqrt{2}$, this means $\tilde{\omega} = \omega$. The two frequencies match.